# MRDS Data Download and Preprocessing

**Purpose:** Clean and filter USGS MRDS records for Zimbabwe, keeping only gold-related deposits
(Au, As, Sb commodity signature), and export deposit point locations as GeoJSON training labels
for the CNN prospectivity model.

**Inputs:**
- `data/raw/mrds.csv` — full MRDS database downloaded from the USGS

**Outputs:**
- `data/processed/mrds_zim_gold.geojson` — cleaned, filtered deposit points (EPSG:4326)

**Why these commodities?**  
Gold deposits in Zimbabwe's Archean greenstone belts express an Au-As-Sb geochemical signature,
with arsenopyrite as the dominant ore mineral. We keep any deposit where gold (Au), arsenic (As),
or antimony (Sb) appears in any commodity field — these pathfinder elements mark the same
hydrothermal system we are mapping.

## 0. Imports and path setup

We use `pathlib.Path` to build file paths relative to the repo root.
This means the notebook works on any machine regardless of where the repo is cloned —
no hardcoded `C:\Users\...` paths.

**How the path logic works:**  
This notebook lives at `mineral-prospectivity-zim/src/data/download_mrds.ipynb`.  
`Path.cwd()` gives us the directory the notebook is running from (i.e. `src/data/`).  
Calling `.parent` once goes up to `src/`.  
Calling `.parent` again goes up to the repo root.  
From there we navigate down into `data/raw/` and `data/processed/`.

In [18]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Navigate from the notebook's directory up to the repo root
REPO_ROOT = Path.cwd().parent.parent

# Input: raw MRDS CSV (downloaded from USGS and placed here manually)
MRDS_RAW = REPO_ROOT / 'data' / 'raw' / 'mrds.csv'

# Output: the cleaned, filtered GeoJSON we are building
OUTPUT_GEOJSON = REPO_ROOT / 'data' / 'processed' / 'mrds_zim_gold.geojson'

# Create the output directory if it does not already exist.
# parents=True: also create any missing parent folders.
# exist_ok=True: do nothing (no error) if the folder already exists.
OUTPUT_GEOJSON.parent.mkdir(parents=True, exist_ok=True)

print(f'Repo root : {REPO_ROOT}')
print(f'Input     : {MRDS_RAW}')
print(f'Output    : {OUTPUT_GEOJSON}')

Repo root : C:\Users\user\OneDrive\Documents\mineral-prospectivity-zim\mineral-prospectivity-zim
Input     : C:\Users\user\OneDrive\Documents\mineral-prospectivity-zim\mineral-prospectivity-zim\data\raw\mrds.csv
Output    : C:\Users\user\OneDrive\Documents\mineral-prospectivity-zim\mineral-prospectivity-zim\data\processed\mrds_zim_gold.geojson


## 1. Load the raw MRDS CSV

`low_memory=False` tells pandas to read the entire column before deciding its data type.
Without it, pandas reads the file in chunks and sometimes guesses the wrong data type
mid-column — for example treating a column as integers until it hits a text value,
then crashing or silently corrupting. With `low_memory=False`, it reads everything
first and makes one correct decision.

In [19]:
df = pd.read_csv(MRDS_RAW, low_memory=False)

print(f'Total MRDS records loaded: {len(df):,}')
print(f'Columns ({len(df.columns)}): {df.columns.tolist()}')

Total MRDS records loaded: 304,632
Columns (46): ['dep_id', 'url', 'mrds_id', 'mas_id', 'site_name', 'latitude', 'longitude', 'region', 'country', 'state', 'county', 'com_type', 'commod1', 'commod2', 'commod3', 'oper_type', 'dep_type', 'prod_size', 'dev_stat', 'ore', 'gangue', 'other_matl', 'orebody_fm', 'work_type', 'model', 'alteration', 'conc_proc', 'names', 'ore_ctrl', 'reporter', 'hrock_unit', 'hrock_type', 'arock_unit', 'arock_type', 'structure', 'tectonic', 'ref', 'yfp_ba', 'yr_fst_prd', 'ylp_ba', 'yr_lst_prd', 'dy_ba', 'disc_yr', 'prod_yrs', 'discr', 'score']


## 2. Filter to Zimbabwe

`df['country'] == 'Zimbabwe'` produces a **boolean Series** — a column of True/False
values, one per row. Passing that into `df[...]` keeps only the rows where the value is True.

`.copy()` creates a fully independent copy of those rows.
Without it, `df_zim` would still be referencing memory inside the original `df`.
If we then modified `df_zim`, pandas would raise a `SettingWithCopyWarning` because
it cannot safely decide whether you want to change the original data or just the slice.
Calling `.copy()` resolves this ambiguity immediately and cleanly.

In [20]:
df_zim = df[df['country'] == 'Zimbabwe'].copy()

print(f'Zimbabwe records: {len(df_zim):,}')

Zimbabwe records: 262


## 3. Filter to gold-related deposits (Au, As, Sb)

MRDS stores commodities across three separate columns: `commod1`, `commod2`, `commod3`.
A deposit may list gold as its primary commodity in `commod1`, or as a secondary
commodity in `commod2` or `commod3`. We must check all three.

**Breaking down `str.contains(...)`:**
- `.str` — tells pandas to treat each cell as a string
- `.contains('Au|As|Sb')` — checks if any of Au, As, or Sb appear anywhere in the cell.
  The `|` is a regex OR operator: it matches Au, or As, or Sb.
- `case=False` — makes the match case-insensitive ('AU', 'Au', 'au' all match)
- `na=False` — if the cell is empty (NaN), return False instead of crashing

**Combining the masks:**  
Each `mask_commod*` is a column of True/False values.  
`|` between two boolean Series means OR: a row is True if it is True in either Series.  
We want rows that match in *any* of the three columns, so we chain OR across all three.

In [21]:
TARGET_COMMODITIES = 'Au|As|Sb|Gold|Arsenic|Antimony|Silver'

# Build one boolean mask per commodity column
mask_commod1 = df_zim['commod1'].str.contains(TARGET_COMMODITIES, case=False, na=False)
mask_commod2 = df_zim['commod2'].str.contains(TARGET_COMMODITIES, case=False, na=False)
mask_commod3 = df_zim['commod3'].str.contains(TARGET_COMMODITIES, case=False, na=False)

# Keep a row if it matches in ANY of the three columns (OR logic)
gold_mask = mask_commod1 | mask_commod2 | mask_commod3

df_gold = df_zim[gold_mask].copy()

print(f'Gold-related records (Au/As/Sb): {len(df_gold):,}')
print()
print('Primary commodity breakdown (commod1):')
print(df_gold['commod1'].value_counts().head(15))

Gold-related records (Au/As/Sb): 73

Primary commodity breakdown (commod1):
commod1
Gold                              48
Asbestos                           9
Gold, Copper                       2
Copper                             2
Silver, Gold, Copper               1
Platinum, Nickel                   1
Platinum, Palladium                1
Tungsten, Silver, Gold, Copper     1
Name: count, dtype: int64


## 4. Select only the columns we need

The full MRDS has many columns we will never use (mining history, references, etc.).
Keeping only what we need reduces memory usage and produces a smaller, cleaner GeoJSON.

We guard against missing columns with a list comprehension that checks each column
against what is actually in the DataFrame — because different MRDS exports
sometimes omit certain fields, and a `KeyError` here would be confusing.

In [22]:
KEEP_COLS = [
    'dep_id',        # MRDS unique deposit identifier
    'site_name',     # Name of the deposit or mine
    'country',       # Should all be Zimbabwe after our filter
    'state',         # Province/region within Zimbabwe
    'commod1',       # Primary commodity
    'commod2',       # Secondary commodity
    'commod3',       # Tertiary commodity
    'dep_type',        # Deposit type (e.g. 'lode gold', 'greenstone belt')
    'latitude',  # Latitude in decimal degrees (WGS84)
    'longitude', # Longitude in decimal degrees (WGS84)
]

# Only keep columns that actually exist in this MRDS export
available_cols = [c for c in KEEP_COLS if c in df_gold.columns]
missing_cols   = [c for c in KEEP_COLS if c not in df_gold.columns]

if missing_cols:
    print(f'WARNING: expected columns not found in file: {missing_cols}')
    print('The script will continue, but check your MRDS version.')

df_gold = df_gold[available_cols].copy()

print(f'Columns kept: {available_cols}')
df_gold.head()

Columns kept: ['dep_id', 'site_name', 'country', 'state', 'commod1', 'commod2', 'commod3', 'dep_type', 'latitude', 'longitude']


,dep_id,site_name,country,state,commod1,commod2,commod3,dep_type,latitude,longitude
21015,10021411,Blanket Mine,Zimbabwe,NaN,Gold,NaN,NaN,NaN,-18.99706,29.87320
21016,10021412,Freda/Rebecca Mine,Zimbabwe,NaN,Gold,NaN,NaN,NaN,-18.99706,29.87320
21017,10021413,Arcturus/Mazase/Munel/Inya Mine,Zimbabwe,NaN,"Silver, Gold, Copper",NaN,NaN,NaN,-18.99706,29.87320
21018,10021414,Renco Mine,Zimbabwe,NaN,"Gold, Copper",NaN,NaN,NaN,-18.99706,29.87320
38204,10039238,Athens Mine,Zimbabwe,NaN,"Gold, Copper",Silver,"Platinum, Nickel",NaN,-19.31361,30.58468


## 5. Validate and clean coordinates

Two cleaning steps before we build geometry:

**Step A — Drop rows with missing coordinates.**  
A deposit with no lat/lon cannot be placed on a map and cannot serve as a training label.
`dropna(subset=[...])` removes rows where either of the listed columns is NaN,
leaving all other columns untouched.

**Step B — Drop rows with coordinates outside Zimbabwe's bounding box.**  
The MRDS has known data entry errors. Some records have coordinates of `0.0, 0.0`
(the default when coordinates were unknown), or coordinates from the wrong country.
Zimbabwe lies approximately within:
- Latitude: -22.42° to -15.61° (negative = south of the equator)
- Longitude: 25.24° to 33.07°

Any point outside this box is a bad record and must be discarded.

The bounding box mask uses `&` (bitwise AND): all four conditions must be True
simultaneously for a row to be kept.

In [24]:
n_before = len(df_gold)

# Step A: drop rows where either coordinate is NaN
df_gold = df_gold.dropna(subset=['latitude', 'longitude'])
n_after_nan = len(df_gold)
print(f'Dropped {n_before - n_after_nan} rows with missing coordinates.')

# Step B: Zimbabwe bounding box
LAT_MIN, LAT_MAX = -22.42, -15.61
LON_MIN, LON_MAX =  25.24,  33.07

# All four conditions must be True (AND logic), so we use & between each one.
# Parentheses around each condition are required when using & in pandas —
# without them Python evaluates & before == and gives wrong results.
in_bbox = (
    (df_gold['latitude']  >= LAT_MIN) &
    (df_gold['latitude']  <= LAT_MAX) &
    (df_gold['longitude'] >= LON_MIN) &
    (df_gold['longitude'] <= LON_MAX)
)

df_gold = df_gold[in_bbox].copy()
n_after_bbox = len(df_gold)
print(f'Dropped {n_after_nan - n_after_bbox} rows with coordinates outside Zimbabwe.')
print(f'Clean records remaining: {n_after_bbox}')

Dropped 0 rows with missing coordinates.
Dropped 0 rows with coordinates outside Zimbabwe.
Clean records remaining: 73


## 6. Build geometry and create a GeoDataFrame

A **GeoDataFrame** is a pandas DataFrame with an extra `geometry` column that stores
geographic shapes — in our case, points.

**`Point(longitude, latitude)` — order matters.**  
Shapely's `Point` takes `(x, y)`, and in geographic coordinates x = longitude, y = latitude.
Getting this backwards is a very common mistake that silently places every deposit
in the ocean (reflected across the diagonal). Always: longitude first, latitude second.

**`apply(..., axis=1)`**  
`axis=1` means run the function once per *row* (as opposed to `axis=0` which runs per column).
The `lambda row: ...` receives the entire row as a Series, so we can access any column by name.

**`crs='EPSG:4326'`**  
EPSG:4326 is the WGS84 geographic coordinate system — the same one GPS uses.
Declaring it here means every downstream tool (GEE, QGIS, rasterio) knows exactly
how to interpret the coordinates without guessing.

In [25]:
# Build a Point geometry for every row: Point(longitude, latitude)
df_gold['geometry'] = df_gold.apply(
    lambda row: Point(row['longitude'], row['latitude']),
    axis=1
)

# Convert to GeoDataFrame
# geometry='geometry' tells geopandas which column holds the shapes
# crs='EPSG:4326' declares WGS84 geographic coordinates
gdf = gpd.GeoDataFrame(df_gold, geometry='geometry', crs='EPSG:4326')

print(f'GeoDataFrame created: {len(gdf)} features')
print(f'CRS: {gdf.crs}')
gdf.head()

GeoDataFrame created: 73 features
CRS: EPSG:4326


,dep_id,site_name,country,state,commod1,commod2,commod3,dep_type,latitude,longitude,geometry
21015,10021411,Blanket Mine,Zimbabwe,NaN,Gold,NaN,NaN,NaN,-18.99706,29.87320,POINT (29.8732 -18.99706)
21016,10021412,Freda/Rebecca Mine,Zimbabwe,NaN,Gold,NaN,NaN,NaN,-18.99706,29.87320,POINT (29.8732 -18.99706)
21017,10021413,Arcturus/Mazase/Munel/Inya Mine,Zimbabwe,NaN,"Silver, Gold, Copper",NaN,NaN,NaN,-18.99706,29.87320,POINT (29.8732 -18.99706)
21018,10021414,Renco Mine,Zimbabwe,NaN,"Gold, Copper",NaN,NaN,NaN,-18.99706,29.87320,POINT (29.8732 -18.99706)
38204,10039238,Athens Mine,Zimbabwe,NaN,"Gold, Copper",Silver,"Platinum, Nickel",NaN,-19.31361,30.58468,POINT (30.58468 -19.31361)


## 7. Final inspection before export

Always do a sanity check before writing the output file.
We check the coordinate ranges on the actual geometry objects,
and print a summary so we have a human-readable record of what went into this file.

`.bounds` returns a DataFrame with columns `minx, miny, maxx, maxy` for every geometry.  
`.agg(...)` aggregates across all rows — we take the minimum of all minx values,
the maximum of all maxx values, etc., to get the overall spatial extent.

In [28]:
bounds = gdf.geometry.bounds.agg({'minx': 'min', 'miny': 'min', 'maxx': 'max', 'maxy': 'max'})

print('=== Final dataset summary ===')
print(f'Total deposit points : {len(gdf)}')
print(f'Longitude range      : {bounds["minx"]:.4f} to {bounds["maxx"]:.4f}')
print(f'Latitude range       : {bounds["miny"]:.4f} to {bounds["maxy"]:.4f}')
print()
print('Deposit type breakdown (dep_type):')
print(gdf['dep_type'].value_counts().head(10))
print()
print('Province breakdown (state):')
print(gdf['state'].value_counts())

=== Final dataset summary ===
Total deposit points : 73
Longitude range      : 28.5783 to 32.7096
Latitude range       : -21.0037 to -16.7971

Deposit type breakdown (dep_type):
dep_type
Unknown           1
VEIN/STOCKWORK    1
Name: count, dtype: int64

Province breakdown (state):
state
Midlands      11
Manicaland     4
Name: count, dtype: int64


## 8. Export to GeoJSON

GeoJSON is the standard format for geographic vector data in JSON.
It is human-readable, natively supported by GEE, QGIS, and every major GIS tool,
and easy to version-control in git because it is plain text.

`driver='GeoJSON'` explicitly tells geopandas the output format.
It is good practice to be explicit rather than relying on the file extension,
because geopandas supports many drivers (Shapefile, GeoPackage, etc.) and
an ambiguous extension could cause a silent format mismatch.

If the output file already exists from a previous run, `to_file` overwrites it —
which is exactly what we want in a reproducible data pipeline.

In [29]:
gdf.to_file(OUTPUT_GEOJSON, driver='GeoJSON')

# Confirm the file was written and show its size on disk
file_size_kb = OUTPUT_GEOJSON.stat().st_size / 1024
print(f'Saved to  : {OUTPUT_GEOJSON}')
print(f'File size : {file_size_kb:.1f} KB')
print(f'Features  : {len(gdf)}')

Saved to  : C:\Users\user\OneDrive\Documents\mineral-prospectivity-zim\mineral-prospectivity-zim\data\processed\mrds_zim_gold.geojson
File size : 23.3 KB
Features  : 73


In [30]:
# Release the GeoDataFrame and all intermediate DataFrames from memory.
# This is good practice at the end of a notebook — especially on a machine
# with limited RAM — because Jupyter keeps all variables alive until you
# explicitly delete them or restart the kernel.

del df, df_zim, df_gold, gdf

# Force Python's garbage collector to release the freed memory immediately
# rather than waiting for it to do so on its own schedule.
import gc
gc.collect()

print('All dataframes released. Notebook complete.')
print(f'Output saved to: {OUTPUT_GEOJSON}')

All dataframes released. Notebook complete.
Output saved to: C:\Users\user\OneDrive\Documents\mineral-prospectivity-zim\mineral-prospectivity-zim\data\processed\mrds_zim_gold.geojson
